In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PowerTransformer
import pingouin as pg

import patsy

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

warnings.filterwarnings('ignore')


In [2]:
# 读取数据
df1 = pd.read_excel(
    r'E:\pycharm all files\眼动数据处理\GSR\完整SCL处理分析\文件提取&GSR分解&去除异常值&绘图\GSR_SCL_AllEvents_pre1sBaseline_3sd.xlsx')
df1.drop('orig_index', axis=1, inplace=True)
df1

,组别,姓名,飞行天数,阶段,data
0,A,付瑞晗,2,1,-1.094561e+05
1,A,付瑞晗,2,1,-1.095295e+05
2,A,付瑞晗,2,1,-1.096041e+05
3,A,付瑞晗,2,1,-1.096799e+05
4,A,付瑞晗,2,1,-1.097564e+05
...,...,...,...,...,...
399461,A,黄博文,7,qifei,-2.959291e+06
399462,A,黄博文,7,qifei,-2.959293e+06
399463,A,黄博文,7,qifei,-2.959295e+06
399464,A,黄博文,7,qifei,-2.959296e+06


In [3]:
# 重置列名
df1.columns = ['组别', '受试者', '飞行天数', '阶段', 'SCL']
# 将阶段为喝酒的行删掉
# df1 = df1[df1['阶段'] != 'hejiu']

# 重命名阶段
df1['阶段'] = df1['阶段'].replace(['1', '2', '3', '4'], '转弯')
df1['阶段'] = df1['阶段'].replace('qifei', '起飞')
df1['阶段'] = df1['阶段'].replace('jiangluo', '降落')
df1['阶段'] = df1['阶段'].replace('hejiu', '喝酒')
df1["组别"] = df1["组别"].replace({
    "A": "Alcohol",
    "B": "Control"
})

df1

,组别,受试者,飞行天数,阶段,SCL
0,Alcohol,付瑞晗,2,转弯,-1.094561e+05
1,Alcohol,付瑞晗,2,转弯,-1.095295e+05
2,Alcohol,付瑞晗,2,转弯,-1.096041e+05
3,Alcohol,付瑞晗,2,转弯,-1.096799e+05
4,Alcohol,付瑞晗,2,转弯,-1.097564e+05
...,...,...,...,...,...
399461,Alcohol,黄博文,7,起飞,-2.959291e+06
399462,Alcohol,黄博文,7,起飞,-2.959293e+06
399463,Alcohol,黄博文,7,起飞,-2.959295e+06
399464,Alcohol,黄博文,7,起飞,-2.959296e+06


In [4]:
# df_one1 = df1[(df1['受试者'] == '王子铭') & (df1['飞行天数'] == 4) & (df1['阶段'] == '喝酒')]
# df_one1

In [5]:
# df_one2 = df1[(df1['受试者'] == '董嘉乐') & (df1['飞行天数'] == 4) & (df1['阶段'] == '起飞')]
# df_one2

In [6]:
# 按阶段、组别、受试者、飞行天数分组，计算均值和中位数
df = df1.groupby(['阶段', '组别', '受试者', '飞行天数'], as_index=False)['SCL'].agg(
    SCL均值='mean',
    SCL中位数='median'
)

df

,阶段,组别,受试者,飞行天数,SCL均值,SCL中位数
0,喝酒,Alcohol,付瑞晗,2,9.939617e+05,9.900174e+05
1,喝酒,Alcohol,付瑞晗,3,9.677228e+05,9.793953e+05
2,喝酒,Alcohol,付瑞晗,4,6.977639e+05,6.971006e+05
3,喝酒,Alcohol,付瑞晗,5,1.734446e+06,1.687678e+06
4,喝酒,Alcohol,付瑞晗,6,3.375287e+05,1.551605e+06
...,...,...,...,...,...,...
762,降落,Control,陈妍,3,-2.482281e+04,-2.479671e+04
763,降落,Control,陈妍,4,-1.171913e+06,-1.171677e+06
764,降落,Control,陈妍,5,-5.726490e+08,-5.726493e+08
765,降落,Control,陈妍,6,4.626787e+04,4.732475e+04


# 三阶模型模型(包含静息阶段无ln转换)

In [8]:
# 1. 对 SCL均值 拟合混合效应模型
# -----------------------------
# 固定效应：组别、飞行天数、阶段
# 随机效应：受试者
model_mean = smf.mixedlm(
    "Q('SCL均值') ~ 组别 * 飞行天数 * 阶段",
    df,
    groups=df["受试者"]
).fit(reml=False)

print("===== SCL均值 混合效应模型结果 =====")
print(model_mean.summary())

# 2. 对 SCL中位数 拟合混合效应模型
# -----------------------------
model_median = smf.mixedlm(
    "Q('SCL中位数') ~ 组别 * 飞行天数 * 阶段",
    df,
    groups=df["受试者"]
).fit(reml=False)

print("\n===== SCL_中位数 混合效应模型结果 =====")
print(model_median.summary())


# 将结果转为 DataFrame
# -----------------------------
def result_to_df(result, label):
    df_out = pd.DataFrame({
        "变量": result.params.index,
        "Coef.": result.params.values,
        "Std.Err.": result.bse.values,
        "z": result.tvalues,
        "P>|z|": result.pvalues,
    })
    df_out["[0.025"] = result.conf_int()[0].values
    df_out["0.975]"] = result.conf_int()[1].values
    df_out["模型"] = label
    df_out["显著性"] = df_out["P>|z|"].apply(lambda p: "显著" if p < 0.05 else "不显著")
    return df_out


df_mean = result_to_df(model_mean, "SCL均值")
df_median = result_to_df(model_median, "SCL中位数")

# 合并
df_all = pd.concat([df_mean, df_median], ignore_index=True)

# -----------------------------
# 保存到 Excel
# -----------------------------
with pd.ExcelWriter("三阶模型模型(包含静息阶段无ln转换)结果.xlsx") as writer:
    df_all.to_excel(writer, sheet_name="全部结果", index=False)
    df_all[df_all["显著性"] == "显著"].to_excel(writer, sheet_name="显著结果", index=False)

print("模型结果已保存到 Excel 文件：混合效应模型结果.xlsx")

===== SCL均值 混合效应模型结果 =====
                                  Mixed Linear Model Regression Results
Model:                        MixedLM             Dependent Variable:             Q('SCL均值')             
No. Observations:             767                 Method:                         ML                     
No. Groups:                   32                  Scale:                          560395465306583424.0000
Min. group size:              23                  Log-Likelihood:                 -16765.3137            
Max. group size:              24                  Converged:                      No                     
Mean group size:              24.0                                                                       
---------------------------------------------------------------------------------------------------------
                                   Coef.            Std.Err.     z    P>|z|     [0.025         0.975]    
-----------------------------------------------------

# 三阶模型模型(包含静息阶段zscore转换)

In [7]:
# 标准化 SCL均值和 SCL中位数
scaler = StandardScaler()
df[['SCL均值_z', 'SCL中位数_z']] = scaler.fit_transform(df[['SCL均值', 'SCL中位数']])

# 混合效应模型（SCL均值_z）
model_mean = smf.mixedlm(
    "SCL均值_z ~ 组别 * 阶段 * 飞行天数",
    df,
    groups=df["受试者"]
)
result_mean = model_mean.fit()
print(result_mean.summary())

# 混合效应模型（SCL中位数_z）
model_median = smf.mixedlm(
    "SCL中位数_z ~ 组别 * 阶段 * 飞行天数",
    df,
    groups=df["受试者"]
)
result_median = model_median.fit()
print(result_median.summary())


# 结果保存到excel
# 提取结果为 DataFrame
def extract_result(result, model_name):
    summary_df = pd.DataFrame({
        "coef": result.params,
        "std_err": result.bse,
        "z_value": result.tvalues,
        "p_value": result.pvalues
    })
    summary_df["显著性"] = summary_df["p_value"].apply(lambda p: "显著" if p < 0.05 else "不显著")
    return summary_df


# 保留模型结果
df_mean = extract_result(result_mean, "SCL均值模型")
df_median = extract_result(result_median, "SCL中位数模型")

# 提取显著结果
df_median_sig = df_median[df_median["显著性"] == "显著"].copy()
df_median_sig["模型"] = "SCL中位数模型"

df_mean_sig = df_mean[df_mean["显著性"] == "显著"].copy()
df_mean_sig["模型"] = "SCL均值模型"

# 合并显著结果
df_sig_all = pd.concat([df_median_sig, df_mean_sig], axis=0)

# 保存到 Excel
with pd.ExcelWriter("三阶模型模型(包含静息阶段zscore转换)结果.xlsx") as writer:
    # 中位数结果
    df_median.to_excel(writer, sheet_name="SCL中位数模型", index=True)

    # 均值结果
    df_mean.to_excel(writer, sheet_name="SCL均值模型", index=True)

    # 显著结果汇总（合并两个模型）
    df_sig_all.to_excel(writer, sheet_name="显著结果汇总", index=True)

                Mixed Linear Model Regression Results
Model:                  MixedLM     Dependent Variable:     SCL均值_z   
No. Observations:       767         Method:                 REML      
No. Groups:             32          Scale:                  1.0053    
Min. group size:        23          Log-Likelihood:         -1110.5815
Max. group size:        24          Converged:              Yes       
Mean group size:        24.0                                          
----------------------------------------------------------------------
                            Coef.  Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------------------
Intercept                    0.047    0.289  0.161 0.872 -0.520  0.613
组别[T.Control]               -0.454    0.409 -1.110 0.267 -1.255  0.348
阶段[T.起飞]                    -0.054    0.408 -0.133 0.894 -0.854  0.745
阶段[T.转弯]                    -0.002    0.408 -0.005 0.996 -0.801  0.798
阶段[T.降落]               

# 三阶模型模型(包含静息阶段yeo-johnson转换)-论文里用这个好了

In [9]:
# 使用yeo-johnson方法进行数据转换(数据分布变成近似正态分布)
pt = PowerTransformer(method='yeo-johnson')
df[['SCL_median_yj', 'SCL_mean_yj']] = pt.fit_transform(df[['SCL中位数', 'SCL均值']])

# 混合效应模型（SCL均值_z）
model_mean = smf.mixedlm(
    "SCL_mean_yj ~ 组别 * 阶段 * 飞行天数",
    df,
    groups=df["受试者"]
)
result_mean = model_mean.fit(method="powell", maxiter=500)
print(result_mean.summary())

# 混合效应模型（SCL中位数_z）
model_median = smf.mixedlm(
    "SCL_median_yj ~ 组别 * 阶段 * 飞行天数",
    df,
    groups=df["受试者"]
)
result_median = model_median.fit(method="powell", maxiter=500)
print(result_median.summary())


# 结果保存到excel
# 提取结果为 DataFrame
def extract_result(result, model_name):
    summary_df = pd.DataFrame({
        "coef": result.params,
        "std_err": result.bse,
        "z_value": result.tvalues,
        "p_value": result.pvalues
    })
    summary_df["显著性"] = summary_df["p_value"].apply(lambda p: "显著" if p < 0.05 else "不显著")
    return summary_df


# 保留模型结果
df_mean = extract_result(result_mean, "SCL均值模型")
df_median = extract_result(result_median, "SCL中位数模型")

# 提取显著结果
df_median_sig = df_median[df_median["显著性"] == "显著"].copy()
df_median_sig["模型"] = "SCL中位数模型"

df_mean_sig = df_mean[df_mean["显著性"] == "显著"].copy()
df_mean_sig["模型"] = "SCL均值模型"

# 合并显著结果
df_sig_all = pd.concat([df_median_sig, df_mean_sig], axis=0)

# 保存到 Excel
with pd.ExcelWriter("三阶模型模型(包含静息阶段yeo-johnson转换)结果.xlsx") as writer:
    # 中位数结果
    df_median.to_excel(writer, sheet_name="SCL中位数模型", index=True)

    # 均值结果
    df_mean.to_excel(writer, sheet_name="SCL均值模型", index=True)

    # 显著结果汇总（合并两个模型）
    df_sig_all.to_excel(writer, sheet_name="显著结果汇总", index=True)

                Mixed Linear Model Regression Results
Model:                 MixedLM     Dependent Variable:     SCL_mean_yj
No. Observations:      767         Method:                 REML       
No. Groups:            32          Scale:                  0.9983     
Min. group size:       23          Log-Likelihood:         -1107.6796 
Max. group size:       24          Converged:              Yes        
Mean group size:       24.0                                           
----------------------------------------------------------------------
                            Coef.  Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------------------
Intercept                    0.009    0.288  0.031 0.975 -0.555  0.574
组别[T.Control]               -0.311    0.407 -0.764 0.445 -1.109  0.487
阶段[T.起飞]                    -0.044    0.406 -0.109 0.913 -0.841  0.752
阶段[T.转弯]                    -0.004    0.406 -0.011 0.992 -0.801  0.792
阶段[T.降落]               

In [11]:
df.to_excel("GSR_SCL绘制热力图的数据.xlsx", index=False)

# 优化器收敛效果测试

In [8]:
model = smf.mixedlm(
    "SCL_mean_yj ~ 组别 * 阶段 * 飞行天数",
    data=df,
    groups=df["受试者"]
)
# result = model.fit(method="lbfgs") #不收敛
result = model.fit(method="powell", maxiter=500)
# result = model.fit(method="bfgs", reml=True, options={"maxiter": 500, "gtol": 1e-6}) #不收敛
# result = model.fit(method="cg", reml=True, options={"maxiter": 1000, "gtol": 1e-6}) #不收敛
print(result.summary())


                Mixed Linear Model Regression Results
Model:                 MixedLM     Dependent Variable:     SCL_mean_yj
No. Observations:      767         Method:                 REML       
No. Groups:            32          Scale:                  0.9983     
Min. group size:       23          Log-Likelihood:         -1107.6796 
Max. group size:       24          Converged:              Yes        
Mean group size:       24.0                                           
----------------------------------------------------------------------
                            Coef.  Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------------------
Intercept                    0.009    0.288  0.031 0.975 -0.555  0.574
组别[T.Control]               -0.311    0.407 -0.764 0.445 -1.109  0.487
阶段[T.起飞]                    -0.044    0.406 -0.109 0.913 -0.841  0.752
阶段[T.转弯]                    -0.004    0.406 -0.011 0.992 -0.801  0.792
阶段[T.降落]               

# yeo-johnson 转换+z-score 标准化后的三阶交互模型结果显著性

In [9]:
# # 假设 df 已经包含 Yeo-Johnson 变换后的 SCL_median_yj
# 
# # 1️⃣ z-score 标准化
# scaler = StandardScaler()
# df['SCL_median_yj_z'] = scaler.fit_transform(df[['SCL_median_yj']])
# 
# # 2️⃣ 建立简化混合效应模型
# # - 固定效应：组别*阶段（二阶交互，去掉飞行天数交互以简化模型）
# # - 随机效应：每个被试姓名随机截距
# md_z = smf.mixedlm(
#     "SCL_median_yj_z ~ 组别*阶段*飞行天数",
#     data=df,
#     groups=df["受试者"]
# )
# 
# # 3️⃣ 拟合模型
# result_z = md_z.fit(method="powell", maxiter=500)
# 
# # 4️⃣ 输出结果
# print(result_z.summary())
# 
# # 1️⃣ z-score 标准化
# scaler = StandardScaler()
# df['SCL_mean_yj_z'] = scaler.fit_transform(df[['SCL_mean_yj']])
# 
# # 2️⃣ 建立简化混合效应模型（随机截距 + 组别*阶段）
# md_mean_z = smf.mixedlm(
#     "SCL_mean_yj_z ~ 组别*阶段*飞行天数",
#     data=df,
#     groups=df["受试者"]
# )
# 
# # 3️⃣ 拟合模型
# result_mean_z = md_mean_z.fit(method="powell", maxiter=500)
# 
# # 4️⃣ 输出结果
# print(result_mean_z.summary())
# 
# 
# # 提取结果函数
# def extract_result(result, model_name):
#     summary_df = pd.DataFrame({
#         "coef": result.params,
#         "std_err": result.bse,
#         "z_value": result.tvalues,
#         "p_value": result.pvalues
#     })
#     summary_df["显著性"] = summary_df["p_value"].apply(lambda p: "显著" if p < 0.05 else "不显著")
#     summary_df["模型"] = model_name
#     return summary_df
# 
# 
# # 提取模型结果
# df_median = extract_result(result_z, "SCL中位数模型")
# df_mean = extract_result(result_mean_z, "SCL均值模型")
# 
# # 提取显著结果
# df_sig_all = pd.concat([
#     df_median[df_median["显著性"] == "显著"],
#     df_mean[df_mean["显著性"] == "显著"]
# ], axis=0)
# 
# # 保存 Excel
# with pd.ExcelWriter("yeo-johnson_zscore_模型显著性结果.xlsx") as writer:
#     df_median.to_excel(writer, sheet_name="SCL中位数模型", index=True)
#     df_mean.to_excel(writer, sheet_name="SCL均值模型", index=True)
#     df_sig_all.to_excel(writer, sheet_name="显著结果汇总", index=True)
